In [ ]:
# 1. Instalar las dependencias necesarias de Unsloth y optimizadores
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# 2. Cargar el modelo base optimizado (Llama-3 8B en 4 bits)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detección
load_in_4bit = True # Usar cuantización de 4 bits para ahorrar memoria

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Configurar el LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # El rango del LoRA (16 o 32 son ideales)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimizado a 0 para mayor velocidad
    bias = "none",    # Optimizado para ahorrar memoria
    use_gradient_checkpointing = "unsloth", # Ahorra mucha VRAM
    random_state = 3407,
)

print("¡Modelo base y estructura LoRA cargados con éxito para Mikeaas!")

In [ ]:
from datasets import Dataset
import json

# 1. Cargar tu archivo JSON (asegúrate de que se llame 'dataset_mikeaas.json' o cambia el nombre aquí)
with open("dataset_mikeaas.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Plantilla que Llama-3-Instruct utiliza para entender el contexto de preguntas y respuestas
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Token de finalización

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Unimos todo usando la plantilla
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Convertimos tu JSON al formato de dataset de Hugging Face
dataset = Dataset.from_list(raw_data)
dataset = dataset.map(formatting_prompts_func, batched = True)

print(f"Dataset cargado correctamente con {len(dataset)} ejemplos.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can speed up training for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Puedes subir esto a 100 o 150 si tienes más de 100 ejemplos
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# ¡A entrenar!
train_stats = trainer.train()
print("¡Entrenamiento del LoRA de Mikeaas completado con éxito!")